# 03. Visual Embeddings with DINOv2

This notebook generates DINOv2 embeddings for the final strict dataset. Each artwork image is transformed into a high-dimensional visual representation that will later be used for downstream modeling.

Unlike SigLIP, which is trained with image-text alignment objectives, DINOv2 is a self-supervised vision model optimized for general-purpose visual representation learning. Comparing the two encoders makes it possible to evaluate whether predictive performance depends on the type of visual pretraining.

In [1]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModel

/home/agrupa-lab/agrupa/IE_capstones/Naji/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ----------------------------
# Configuration
# ----------------------------
DATASET_CSV = "../Data/Processed/final_dataset_strict.csv"
MODEL_NAME = "facebook/dinov2-base"

OUTPUT_DIR = Path("../Embeddings/dinov2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_EMBEDDINGS = OUTPUT_DIR / "dinov2_strict_embeddings.npy"
OUTPUT_IDS = OUTPUT_DIR / "dinov2_strict_catnos.csv"
OUTPUT_MERGED = OUTPUT_DIR / "final_dataset_strict_with_dinov2.csv"
OUTPUT_FAILED = OUTPUT_DIR / "dinov2_strict_failed_paths.csv"

BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
df = pd.read_csv(DATASET_CSV)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head(2)

Dataset shape: (3198, 19)

Columns:
['cat_no', 'titulo', 'autor', 'escuela_obra', 'tipo_objeto', 'datacion', 'tema', 'is_religious', 'is_fauna', 'century', 'image_path', 'descripcion', 'animal_cluster', 'n_descriptores_fila', 'n_en_diccionario_fila', 'dirmean_Warmth', 'n_dirmean_Warmth', 'dirmean_Competence', 'n_dirmean_Competence']


,cat_no,titulo,autor,escuela_obra,tipo_objeto,datacion,tema,is_religious,is_fauna,century,image_path,descripcion,animal_cluster,n_descriptores_fila,n_en_diccionario_fila,dirmean_Warmth,n_dirmean_Warmth,dirmean_Competence,n_dirmean_Competence
0,P000002,El juicio de Paris,"Albani, Francesco",Italiana,Pintura,1650 - 1660,NaN,0,1,17th c.,obras/P000002.jpg,"La obra de Francesco Albani, uno de los discíp...",purity,890.0,62.0,0.333333,9,0.7,10
1,P000006,Sagrada Familia y el cardenal Fernando de Medici,"Allori, Alessandro",Italiana,Pintura,1584,NaN,1,1,16th c.,obras/P000006.jpg,"San José, la Virgen con el Niño en brazos y Sa...",other,148.0,17.0,0.937500,4,0.5,2


In [4]:
# ----------------------------
# Resolve image paths safely
# ----------------------------
RAW_IMAGE_ROOT = Path("/home/agrupa-lab/agrupa/data_raw")

if "file_path" in df.columns:
    df["resolved_image_path"] = df["file_path"].astype(str)
elif "image_path" in df.columns:
    df["resolved_image_path"] = df["image_path"].apply(
        lambda p: str(RAW_IMAGE_ROOT / str(p)) if pd.notna(p) else None
    )
else:
    raise ValueError("No image column found. Expected 'file_path' or 'image_path'.")

df["file_exists"] = df["resolved_image_path"].apply(
    lambda p: Path(p).exists() if pd.notna(p) else False
)

print("Total rows:", len(df))
print("Missing images:", int((~df["file_exists"]).sum()))

df = df[df["file_exists"]].copy().reset_index(drop=True)
print("Rows used for embeddings:", len(df))

Total rows: 3198
Missing images: 138
Rows used for embeddings: 3060


## 2. Loading the DINOv2 model

DINOv2 is loaded in evaluation mode and used only as a frozen feature extractor. No fine-tuning is performed at this stage. For each image, the CLS token from the final hidden state is used as the embedding representation.

In [5]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(device)
_ = model.eval()

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2614.06it/s]


In [6]:
def load_image(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

## 3. Extracting DINOv2 embeddings in batches

Embeddings are computed in batches to improve efficiency and avoid memory issues. For each valid image, the CLS token from the final hidden state is extracted and stored together with the corresponding `cat_no`.

In [7]:
all_embeddings = []
all_catnos = []
failed_paths = []

for start in tqdm(range(0, len(df), BATCH_SIZE), desc="Extracting DINOv2"):
    batch_df = df.iloc[start:start + BATCH_SIZE]

    images = []
    catnos = []

    for _, row in batch_df.iterrows():
        img = load_image(row["resolved_image_path"])
        if img is None:
            failed_paths.append(row["resolved_image_path"])
            continue
        images.append(img)
        catnos.append(row["cat_no"])

    if len(images) == 0:
        continue

    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    batch_embeddings = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()

    all_embeddings.append(batch_embeddings)
    all_catnos.extend(catnos)

if len(all_embeddings) == 0:
    raise RuntimeError("No embeddings were extracted.")

embeddings = np.vstack(all_embeddings)
ids_df = pd.DataFrame({"cat_no": all_catnos})

print("Embedding matrix shape:", embeddings.shape)
print("Number of IDs:", len(ids_df))
print("Failed image loads:", len(failed_paths))

Extracting DINOv2: 100%|██████████| 192/192 [00:32<00:00,  5.91it/s]

Embedding matrix shape: (3060, 768)
Number of IDs: 3060
Failed image loads: 0


In [8]:
# Save raw embeddings
np.save(OUTPUT_EMBEDDINGS, embeddings)
ids_df.to_csv(OUTPUT_IDS, index=False)

# Save merged dataset with embedding columns
emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=emb_cols)

merged = pd.concat(
    [ids_df.reset_index(drop=True), emb_df.reset_index(drop=True)],
    axis=1
)

final_merged = df.merge(merged, on="cat_no", how="inner")
final_merged.to_csv(OUTPUT_MERGED, index=False)

if failed_paths:
    pd.DataFrame({"failed_path": failed_paths}).to_csv(OUTPUT_FAILED, index=False)

print("\nSaved files:")
print("-", OUTPUT_EMBEDDINGS)
print("-", OUTPUT_IDS)
print("-", OUTPUT_MERGED)
if failed_paths:
    print("-", OUTPUT_FAILED)


Saved files:
- ../Embeddings/dinov2/dinov2_strict_embeddings.npy
- ../Embeddings/dinov2/dinov2_strict_catnos.csv
- ../Embeddings/dinov2/final_dataset_strict_with_dinov2.csv


## 4. Quick validation of saved outputs

Before moving to the modeling stage, the saved outputs are checked to confirm that the number of embeddings, IDs, and merged dataset rows are all aligned correctly.

In [9]:
emb = np.load(OUTPUT_EMBEDDINGS)
ids = pd.read_csv(OUTPUT_IDS)
merged_df = pd.read_csv(OUTPUT_MERGED)

print("Embeddings shape:", emb.shape)
print("IDs shape:", ids.shape)
print("Merged dataset shape:", merged_df.shape)

print("\nUnique cat_no in IDs:", ids["cat_no"].nunique())
print("Unique cat_no in merged dataset:", merged_df["cat_no"].nunique())

print("\nEmbedding stats:")
print("Mean:", emb.mean())
print("Std:", emb.std())
print("Min:", emb.min())
print("Max:", emb.max())

Embeddings shape: (3060, 768)
IDs shape: (3060, 1)
Merged dataset shape: (3060, 789)

Unique cat_no in IDs: 3060
Unique cat_no in merged dataset: 3060

Embedding stats:
Mean: -0.02179339
Std: 1.6559421
Min: -8.454286
Max: 8.208852
